In [322]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
import yaml
import numpy as np
import geopandas as gpd
from pyproj import Transformer
from shapely.geometry import Polygon
from folium import plugins
import requests
import os
from math import radians, cos, sin, asin, sqrt


import calliope

# We increase logging verbosity
calliope.set_log_verbosity("INFO", include_solver_output=False)

In [323]:

# Adjust links costs to a specified value
import pandas as pd

# Load links costs CSV
links_costs_df = pd.read_csv('links_costs.csv')

# Define cost value (user-adjustable)
cost_value = 100000000  # 100000000 for high, 0.01 for low

# Update the cost column
links_costs_adjusted = links_costs_df.copy()
links_costs_adjusted['cost_flow_cap_per_distance'] = cost_value


# Save to CSV
links_costs_adjusted.to_csv('links_costs.csv', index=False)
print(f"  All {len(links_costs_adjusted)} links now have cost: {cost_value:,.2f}")


  All 11 links now have cost: 100,000,000.00


In [324]:
model = calliope.read_yaml('model.yaml')

[2026-03-27 10:24:08] INFO     Math init | loading pre-defined math.
[2026-03-27 10:24:08] INFO     Math init | loading math files {'operate', 'base', 'storage_inter_cluster', 'spores', 'milp'}.
[2026-03-27 10:24:08] INFO     Model: preprocessing data
[2026-03-27 10:24:08] INFO     Math build | building applied math with ['base'].
[2026-03-27 10:24:09] INFO     input data `color` not defined in model math; it will not be available in the optimisation problem.
[2026-03-27 10:24:09] INFO     input data `name` not defined in model math; it will not be available in the optimisation problem.
[2026-03-27 10:24:09] INFO     input data `flow_cap` not defined in model math; it will not be available in the optimisation problem.
[2026-03-27 10:24:09] INFO     input data `link_to` not defined in model math; it will not be available in the optimisation problem.
[2026-03-27 10:24:09] INFO     input data `link_from` not defined in model math; it will not be available in the optimisation problem.
[202

In [325]:
model.inputs

<xarray.Dataset> Size: 536kB
Dimensions:                       (costs: 1, techs: 15, nodes: 12, carriers: 1,
                                   timesteps: 336)
Coordinates:
  * costs                         (costs) object 8B 'monetary'
  * techs                         (techs) object 120B 'HV_station_to_transmis...
  * carriers                      (carriers) object 8B 'electricity'
  * nodes                         (nodes) object 96B 'HV_station' ... 'transm...
  * timesteps                     (timesteps) datetime64[ns] 3kB 2024-09-01 ....
Data variables: (12/35)
    cost_interest_rate            (costs) float64 8B 0.1
    bigM                          float64 8B 1e+06
    objective_cost_weights        (costs) float64 8B 1.0
    base_tech                     (techs) object 120B 'transmission' ... 'tra...
    carrier_in                    (nodes, techs, carriers) bool 180B True ......
    color                         (techs) object 120B '#823739' ... '#823739'
    ...                            ...
    source_use_equals             (techs, timesteps) float64 40kB nan ... nan
    sink_use_equals               (timesteps, techs, nodes) float64 484kB nan...
    definition_matrix             (nodes, techs, carriers) bool 180B True ......
    distance                      (techs) float64 120B 0.05581 nan ... 0.5484
    timestep_resolution           (timesteps) float64 3kB 1.0 1.0 ... 1.0 1.0
    timestep_weights              (timesteps) float64 3kB 1.0 1.0 ... 1.0 1.0

In [326]:
model.inputs.flow_cap_max.to_series().dropna()  

techs
HV_station_to_transmission_1            100000.0
pv                                        1000.0
supply_grid_power                        50000.0
transmission_1_to_transmission_2        100000.0
transmission_1_to_transmission_3        100000.0
transmission_2_to_data_center_3         100000.0
transmission_3_to_data_center_2         100000.0
transmission_3_to_transmission_4        100000.0
transmission_3_to_transmission_5        100000.0
transmission_4_to_data_center_1         100000.0
transmission_5_to_transmission_6        100000.0
transmission_6_to_transmission_7        100000.0
transmission_7_to_residential_demand    100000.0
Name: flow_cap_max, dtype: float64

In [327]:
model.inputs.sink_use_equals.sum(
    "timesteps", min_count=1, skipna=True
).to_series().dropna()

techs               nodes             
demand_electricity  data_center_1         1.680000e+06
                    data_center_2         2.688000e+06
                    data_center_3         4.032000e+06
                    residential_demand    2.233275e+05
Name: sink_use_equals, dtype: float64

In [328]:
model.build(force=True)

[2026-03-27 10:24:09] INFO     Model: backend build starting
[2026-03-27 10:24:09] INFO     Optimisation Model | parameters/lookups | Generated.
[2026-03-27 10:24:10] INFO     Optimisation Model | variables | Generated.
[2026-03-27 10:24:11] INFO     Optimisation Model | global_expressions | Generated.
[2026-03-27 10:24:13] INFO     Optimisation Model | constraints | Generated.
[2026-03-27 10:24:13] INFO     Optimisation Model | piecewise_constraints | Generated.
[2026-03-27 10:24:13] INFO     Optimisation Model | objectives | Generated.
[2026-03-27 10:24:13] INFO     Model: backend build complete


In [329]:
model.backend.parameters

<xarray.Dataset> Size: 534kB
Dimensions:                             (costs: 1, techs: 15, timesteps: 336,
                                         nodes: 12)
Coordinates:
  * costs                               (costs) object 8B 'monetary'
  * techs                               (techs) object 120B 'HV_station_to_tr...
  * nodes                               (nodes) object 96B 'HV_station' ... '...
  * timesteps                           (timesteps) datetime64[ns] 3kB 2024-0...
Data variables: (12/59)
    area_use_max                        float64 8B nan
    area_use_min                        float64 8B nan
    area_use_per_flow_cap               float64 8B nan
    available_area                      float64 8B nan
    bigM                                object 8B parameters[bigM][0]
    cost_flow_cap_per_distance          (costs, techs) object 120B parameters...
    ...                                  ...
    storage_cap_per_unit                float64 8B nan
    storage_discharge_depth             float64 8B nan
    storage_initial                     float64 8B nan
    storage_loss                        (techs) object 120B nan ... nan
    timestep_resolution                 (timesteps) object 3kB parameters[tim...
    timestep_weights                    (timesteps) object 3kB parameters[tim...

In [330]:
model.solve(solver='gurobi')

[2026-03-27 10:24:14] INFO     Optimisation model | starting model in base mode.
[2026-03-27 10:24:16] INFO     Backend: solver finished running. Time since start of solving optimisation problem: 0:00:02.358892
[2026-03-27 10:24:16] INFO     Postprocessing: applied zero threshold 1e-10 to model results.
[2026-03-27 10:24:16] INFO     Postprocessing: ended. Time since start of solving optimisation problem: 0:00:02.420766
[2026-03-27 10:24:16] INFO     Backend: model solve completed. Time since start of solving optimisation problem: 0:00:02.420766


In [331]:
model.backend.parameters

<xarray.Dataset> Size: 534kB
Dimensions:                             (costs: 1, techs: 15, timesteps: 336,
                                         nodes: 12)
Coordinates:
  * costs                               (costs) object 8B 'monetary'
  * techs                               (techs) object 120B 'HV_station_to_tr...
  * nodes                               (nodes) object 96B 'HV_station' ... '...
  * timesteps                           (timesteps) datetime64[ns] 3kB 2024-0...
Data variables: (12/59)
    area_use_max                        float64 8B nan
    area_use_min                        float64 8B nan
    area_use_per_flow_cap               float64 8B nan
    available_area                      float64 8B nan
    bigM                                object 8B parameters[bigM][0]
    cost_flow_cap_per_distance          (costs, techs) object 120B parameters...
    ...                                  ...
    storage_cap_per_unit                float64 8B nan
    storage_discharge_depth             float64 8B nan
    storage_initial                     float64 8B nan
    storage_loss                        (techs) object 120B nan ... nan
    timestep_resolution                 (timesteps) object 3kB parameters[tim...
    timestep_weights                    (timesteps) object 3kB parameters[tim...

In [332]:
model.results

<xarray.Dataset> Size: 4MB
Dimensions:                      (nodes: 12, techs: 15, carriers: 1,
                                  timesteps: 336, costs: 1)
Coordinates:
  * techs                        (techs) object 120B 'HV_station_to_transmiss...
  * nodes                        (nodes) object 96B 'HV_station' ... 'transmi...
  * carriers                     (carriers) object 8B 'electricity'
  * timesteps                    (timesteps) datetime64[ns] 3kB 2024-09-01 .....
  * costs                        (costs) object 8B 'monetary'
Data variables: (12/24)
    flow_cap                     (nodes, techs, carriers) float64 1kB 2.544e+...
    link_flow_cap                (techs) float64 120B 2.544e+04 nan ... 1.09e+03
    flow_out                     (nodes, techs, carriers, timesteps) float64 484kB ...
    flow_in                      (nodes, techs, carriers, timesteps) float64 484kB ...
    flow_export                  (nodes, techs, carriers, timesteps) float64 484kB ...
    source_use                   (nodes, techs, timesteps) float64 484kB nan ...
    ...                           ...
    min_cost_optimisation        float64 8B 4.962e+09
    capacity_factor              (nodes, techs, carriers, timesteps) float64 484kB ...
    systemwide_capacity_factor   (techs, carriers) float64 120B 0.4987 ... 0....
    systemwide_levelised_cost    (techs, costs, carriers) float64 120B 75.02 ...
    total_levelised_cost         (costs, carriers) float64 8B 572.5
    unmet_sum                    (nodes, carriers, timesteps) float64 32kB 0....

In [333]:
costs = model.results.cost.to_series().dropna()
costs.head()

nodes          techs                            costs   
HV_station     HV_station_to_transmission_1     monetary    3.198059e+08
               supply_grid_power                monetary    1.074948e+02
data_center_1  battery                          monetary    1.690250e-01
               pv                               monetary    4.225625e-02
               transmission_4_to_data_center_1  monetary    2.581909e+08
Name: cost, dtype: float64

In [334]:
lcoes = (
    model.results.systemwide_levelised_cost.sel(carriers="electricity")
    .to_series()
    .dropna()
)
lcoes.head()

techs                             costs   
HV_station_to_transmission_1      monetary    7.501854e+01
battery                           monetary    6.504486e-06
pv                                monetary    9.322571e-07
supply_grid_power                 monetary    1.260202e-05
transmission_1_to_transmission_2  monetary    3.901405e+01
Name: systemwide_levelised_cost, dtype: float64

In [335]:
# We set the color mapping to use in all our plots by extracting the colors defined in the technology definitions of our model.
colors = model.inputs.color.to_series().to_dict()

In [336]:
df_electricity = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers="electricity")
    .sum("nodes")
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_electricity_demand = df_electricity[df_electricity.techs == "demand_electricity"]
df_electricity_other = df_electricity[df_electricity.techs != "demand_electricity"]

print(df_electricity.head())

fig1 = px.bar(
    df_electricity_other,
    x="timesteps",
    y="Flow in/out (kWh)",
    color="techs",
    color_discrete_map=colors,
)
fig1.add_scatter(
    x=df_electricity_demand.timesteps,
    y=-1 * df_electricity_demand["Flow in/out (kWh)"],
    marker_color="black",
    name="demand",
)

                          techs           timesteps  Flow in/out (kWh)
0  HV_station_to_transmission_1 2024-09-01 00:00:00         -14.228798
1  HV_station_to_transmission_1 2024-09-01 01:00:00         -14.234581
2  HV_station_to_transmission_1 2024-09-01 02:00:00         -14.234059
3  HV_station_to_transmission_1 2024-09-01 03:00:00         -14.236125
4  HV_station_to_transmission_1 2024-09-01 04:00:00         -14.235389


In [337]:
carriers = ["electricity"]
df_flows = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers=carriers)
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_demand = df_flows[df_flows.techs.str.contains("demand")]
df_flows_other = df_flows[~df_flows.techs.str.contains("demand")]

print(df_flows.head())

node_order = df_flows_other.nodes.unique()

fig = px.bar(
    df_flows_other,
    x="timesteps",
    y="Flow in/out (kWh)",
    facet_row="nodes",
    facet_col="carriers",
    color="techs",
    category_orders={"nodes": node_order, "carriers": carriers},
    height=1000,
    color_discrete_map=colors,
)

showlegend = True
# we reverse the node order (`[::-1]`) because the rows are numbered from bottom to top.
for row, node in enumerate(node_order[::-1]):
    for col, carrier in enumerate(carriers):
        demand_ = df_demand.loc[
            (df_demand.nodes == node) & (df_demand.techs == f"demand_{carrier}"),
            "Flow in/out (kWh)",
        ]
        if not demand_.empty:
            fig.add_scatter(
                x=model.results.timesteps.values,
                y=-1 * demand_,
                row=row + 1,
                col=col + 1,
                marker_color="black",
                name="Demand",
                legendgroup="demand",
                showlegend=showlegend,
            )
            showlegend = False
fig.update_yaxes(matches=None)
fig.show()

        nodes                         techs     carriers           timesteps  \
0  HV_station  HV_station_to_transmission_1  electricity 2024-09-01 00:00:00   
1  HV_station  HV_station_to_transmission_1  electricity 2024-09-01 01:00:00   
2  HV_station  HV_station_to_transmission_1  electricity 2024-09-01 02:00:00   
3  HV_station  HV_station_to_transmission_1  electricity 2024-09-01 03:00:00   
4  HV_station  HV_station_to_transmission_1  electricity 2024-09-01 04:00:00   

   Flow in/out (kWh)  
0      -25375.413456  
1      -25385.727027  
2      -25384.795205  
3      -25388.480137  
4      -25387.167115  


In [338]:
df_capacity = (
    model.results.flow_cap.where(
        ~model.inputs.base_tech.str.contains("demand|transmission")
    )
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)

print(df_capacity.head())

fig = px.bar(
    df_capacity,
    x="nodes",
    y="Flow capacity (kW)",
    color="techs",
    facet_col="carriers",
    color_discrete_map=colors,
)
fig.show()

           nodes              techs     carriers  Flow capacity (kW)
0     HV_station  supply_grid_power  electricity        25438.787024
1  data_center_1            battery  electricity         1000.000000
2  data_center_1                 pv  electricity         1000.000000
3  data_center_2            battery  electricity         1000.000000
4  data_center_2                 pv  electricity         1000.000000


In [339]:
with open("model.yaml", "r", encoding="utf-8") as f:
    model_def = yaml.safe_load(f)

node_techs = {
    node: list((node_data.get("techs") or {}).keys())
    for node, node_data in model_def.get("nodes", {}).items()
}

In [340]:
# Build a simple system map (nodes + links)
nodes = pd.read_csv("nodes_coordinates.csv")
links = pd.read_csv("links_techs.csv")

flow_cap = (
    model.results.flow_cap.to_series().dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)
flow_cap_lookup = dict(zip(flow_cap["techs"], flow_cap["Flow capacity (kW)"]))

center = [nodes.latitude.mean(), nodes.longitude.mean()]
system_map = folium.Map(location=center, zoom_start=15, tiles="CartoDB voyager")

# Add link lines
for _, row in links.iterrows():
    from_row = nodes.loc[nodes.nodes == row["link_from"]].iloc[0]
    to_row = nodes.loc[nodes.nodes == row["link_to"]].iloc[0]
    capacity = flow_cap_lookup.get(row["techs"], row.get("flow_cap_max"))
    popup = (
        f"<b>{row['techs']}</b><br>From: {row['link_from']}<br>To: {row['link_to']}"
    )
    if capacity is not None:
        popup += f"<br>Capacity: {capacity:.2f} kW"
    folium.PolyLine(
        locations=[[from_row.latitude, from_row.longitude], [to_row.latitude, to_row.longitude]],
        color=row.get("color", "#1f77b4"),
        weight=3,
        opacity=1,
        popup=popup,
    ).add_to(system_map)

# Add node markers
color_map = model.inputs.color.to_series().to_dict()
for _, row in nodes.iterrows():
    node = row["nodes"]
    techs = node_techs.get(node, [])
    base_types = (
        model.inputs.base_tech.sel(techs=techs).to_series().to_dict()
        if techs
        else {}
    )

    node_type = "Other"
    if any(t == "demand" for t in base_types.values()):
        node_type = "Demand"
    elif any(t == "supply" for t in base_types.values()):
        node_type = "Supply"

    marker_color = "#666666"
    for tech in techs:
        if tech in color_map:
            marker_color = color_map[tech]
            break

    popup = f"<b>{node}</b> ({node_type})<br>Techs: {', '.join(techs)}"
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6,
        color=marker_color,
        fill=True,
        fillColor=marker_color,
        fillOpacity=1,
        popup=popup,
    ).add_to(system_map)

system_map.save("system_map.html")

In [341]:
# Export Comprehensive Bill of Materials (All Technologies + Transmission)

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate the great circle distance between two points on earth"""
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371000  # Radius of earth in meters
    return c * r

# ===== Non-transmission Technologies (Supply, Storage, etc.) =====
non_transmission_bom = (
    model.results.flow_cap
    .where(~model.inputs.base_tech.isin(["demand", "transmission"]))
    .to_series()
    .where(lambda x: x > 0)
    .dropna()
    .to_frame("capacity_kw")
    .reset_index()
)
non_transmission_bom = non_transmission_bom.rename(columns={"techs": "technology"})
non_transmission_bom["distance_m"] = None
non_transmission_bom["storage_capacity_kwh"] = None

# ===== Storage Technologies (Battery) =====
# Extract storage capacity for technologies that have it
storage_cap_data = (
    model.results.storage_cap
    .to_series()
    .where(lambda x: x > 0)
    .dropna()
    .to_frame("storage_capacity_kwh")
    .reset_index()
)
storage_cap_data = storage_cap_data.rename(columns={"techs": "technology"})

# For storage, set flow_cap (power) to NaN and distance_m to None
storage_cap_data["capacity_kw"] = None
storage_cap_data["distance_m"] = None

# Merge storage capacity back into non-transmission BOM
non_transmission_bom = non_transmission_bom.merge(
    storage_cap_data,
    on=["nodes", "technology"],
    how="left",
    suffixes=("", "_storage")
)

# Combine storage_capacity_kwh columns (one from initialization, one from merge)
if "storage_capacity_kwh_storage" in non_transmission_bom.columns:
    non_transmission_bom["storage_capacity_kwh"] = (
        non_transmission_bom["storage_capacity_kwh"]
        .where(non_transmission_bom["storage_capacity_kwh"].notna(), 
               non_transmission_bom["storage_capacity_kwh_storage"])
    )
    non_transmission_bom = non_transmission_bom.drop(columns=["storage_capacity_kwh_storage"])

# Update capacity_kw for storage technologies (set 0 to NaN)
non_transmission_bom.loc[non_transmission_bom["capacity_kw"] <= 0, "capacity_kw"] = None

# ===== Transmission Technologies (Power Lines) =====
power_lines_cap = (
    model.results.flow_cap
    .where(model.inputs.base_tech == "transmission")
    .to_series()
    .where(lambda x: x > 0)
    .dropna()
    .to_frame("capacity_kw")
    .reset_index()
)
power_lines_cap = power_lines_cap.groupby("techs", as_index=False)["capacity_kw"].first()

# Load links and coordinates to calculate distances
links_data = pd.read_csv("links_techs.csv")
nodes_coords = pd.read_csv("nodes_coordinates.csv")
coord_dict = dict(zip(nodes_coords['nodes'], zip(nodes_coords['latitude'], nodes_coords['longitude'])))

# Calculate distances
links_data['distance_m'] = links_data.apply(
    lambda row: haversine_distance(
        coord_dict[row['link_from']][0], coord_dict[row['link_from']][1],
        coord_dict[row['link_to']][0], coord_dict[row['link_to']][1]
    ),
    axis=1
)

# Merge transmission capacity with distances
transmission_bom = power_lines_cap.merge(
    links_data[["techs", "distance_m"]],
    left_on="techs",
    right_on="techs",
    how="left"
)
transmission_bom = transmission_bom.rename(columns={"techs": "technology"})

# ===== Combine All Technologies =====
# Ensure transmission_bom has all required columns
if "nodes" not in transmission_bom.columns:
    transmission_bom["nodes"] = None
if "storage_capacity_kwh" not in transmission_bom.columns:
    transmission_bom["storage_capacity_kwh"] = None

# Reorder columns consistently
required_columns = ["nodes", "technology", "capacity_kw", "distance_m", "storage_capacity_kwh"]
non_transmission_bom = non_transmission_bom[required_columns]
transmission_bom = transmission_bom[required_columns]

# Explicitly ensure distance_m and storage_capacity_kwh are float64 before concatenating
non_transmission_bom['distance_m'] = non_transmission_bom['distance_m'].astype('float64')
transmission_bom['distance_m'] = transmission_bom['distance_m'].astype('float64')
non_transmission_bom['capacity_kw'] = non_transmission_bom['capacity_kw'].astype('float64')
transmission_bom['capacity_kw'] = transmission_bom['capacity_kw'].astype('float64')
non_transmission_bom['storage_capacity_kwh'] = non_transmission_bom['storage_capacity_kwh'].astype('float64')
transmission_bom['storage_capacity_kwh'] = transmission_bom['storage_capacity_kwh'].astype('float64')

bom_df = pd.concat([non_transmission_bom, transmission_bom], ignore_index=True)

# Sort by technology name and capacity (handle NaN values)
bom_df = bom_df.sort_values(
    by=["technology", "capacity_kw"],
    ascending=[True, False],
    na_position='last'
).reset_index(drop=True)

# Save to single comprehensive CSV
os.makedirs("outputs", exist_ok=True)
bom_df.to_csv("outputs/bill_of_materials.csv", index=False)
#print(f"\nSaved to outputs/bill_of_materials.csv")

In [342]:

# Extract dispatch data for visualization
import numpy as np

# Extract timesteps
timesteps = model.results.timesteps.values
print(f"Timesteps: {len(timesteps)} values")
#print(timesteps)

# Extract grid power (supply_grid_power flow_out)
grid_power_data = model.results.flow_out.sel(techs="supply_grid_power", carriers="electricity").to_series().dropna()
grid_power = grid_power_data.values
print(f"\nGrid Power: min={grid_power.min():.1f}, max={grid_power.max():.1f} kW")

# Extract PV dispatch (pv flow_out)
pv_data = model.results.flow_out.sel(techs="pv", carriers="electricity").to_series().dropna()
pv_dispatch = pv_data.values
print(f"PV Dispatch: min={pv_dispatch.min():.1f}, max={pv_dispatch.max():.1f} kW")

# Extract battery dispatch (net flow: out - in), summed across all nodes
battery_out = model.results.flow_out.sel(techs="battery", carriers="electricity").to_series().dropna()
battery_in = model.results.flow_in.sel(techs="battery", carriers="electricity").to_series().dropna()
# Sum across all nodes by grouping by timesteps
battery_dispatch_series = (battery_out - battery_in).groupby('timesteps').sum()
battery_dispatch = battery_dispatch_series.values
print(f"Battery Dispatch (total across all nodes): min={battery_dispatch.min():.1f}, max={battery_dispatch.max():.1f} kW")


Timesteps: 336 values

Grid Power: min=24333.3, max=25438.8 kW
PV Dispatch: min=0.0, max=605.2 kW
Battery Dispatch (total across all nodes): min=-1471.3, max=1312.3 kW


In [343]:
# ===== Extract prices from CSV =====
# Load electricity prices from the day_ahead_prices.csv file
try:
    prices_csv = pd.read_csv('day_ahead_prices.csv')
    print("CSV columns:", prices_csv.columns.tolist())
    print("CSV dtypes:\n", prices_csv.dtypes)
    print("\nFull CSV:")
    print(prices_csv)
    
    # The alabala column contains the prices
    if 'alabala' in prices_csv.columns:
        # Convert to numeric
        prices_values = pd.to_numeric(prices_csv['alabala'], errors='coerce')
        # Skip the first row (header/tech info) and get prices for timesteps
        day_ahead_prices_corrected = prices_values[1:].reset_index(drop=True).values[:len(timesteps)]
    else:
        day_ahead_prices_corrected = None
except Exception as e:
    print(f"Error loading prices: {e}")
    import traceback
    traceback.print_exc()
    day_ahead_prices_corrected = None

CSV columns: ['comment', 'alabala']
CSV dtypes:
 comment     object
alabala    float64
dtype: object

Full CSV:
                  comment       alabala
0                   techs           NaN
1     2024-01-01 00:00:00  1.000000e-05
2     2024-01-01 01:00:00  1.000000e-06
3     2024-01-01 02:00:00  0.000000e+00
4     2024-01-01 03:00:00 -1.000000e-06
...                   ...           ...
8779  2024-12-31 19:00:00  7.056000e-03
8780  2024-12-31 20:00:00  5.990000e-03
8781  2024-12-31 21:00:00  1.820000e-03
8782  2024-12-31 22:00:00  9.060000e-04
8783  2024-12-31 23:00:00  5.200000e-05

[8784 rows x 2 columns]


In [344]:
# ===== Improved Plot with Better Visibility =====
# Create a multi-row layout with separate scales for each component
fig_improved = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    subplot_titles=("Grid Power Supply", "PV Generation", "Battery Dispatch", "Spot Prices"),
    row_heights=[0.25, 0.2, 0.2, 0.35],
    vertical_spacing=0.08,
    specs=[[{}], [{}], [{}], [{}]]
)

# === Row 1: Grid Power (zoomed to actual range to show variations) ===
fig_improved.add_trace(
    go.Scatter(
        x=timesteps,
        y=grid_power,
        name="Grid Power",
        mode='lines',
        line=dict(color='#1f77b4', width=2.5),
        fill='tozeroy',
        fillcolor='rgba(31, 119, 180, 0.2)',
        hovertemplate='<b>%{x}</b><br>Grid Power: %{y:.1f} kW<extra></extra>'
    ),
    row=1, col=1
)

# === Row 2: PV Generation (smaller scale) ===
fig_improved.add_trace(
    go.Scatter(
        x=timesteps,
        y=pv_dispatch,
        name="PV",
        mode='lines',
        line=dict(color='#ff7f0e', width=2.5),
        fill='tozeroy',
        fillcolor='rgba(255, 127, 14, 0.2)',
        hovertemplate='<b>%{x}</b><br>PV: %{y:.1f} kW<extra></extra>'
    ),
    row=2, col=1
)

# === Row 3: Battery (can be positive or negative) ===
# Add a zero line for reference
fig_improved.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=3, col=1)

fig_improved.add_trace(
    go.Scatter(
        x=timesteps,
        y=battery_dispatch,
        name="Battery",
        mode='lines',
        line=dict(color='#2ca02c', width=2.5),
        fill='tozeroy',
        fillcolor='rgba(44, 160, 44, 0.2)',
        hovertemplate='<b>%{x}</b><br>Battery: %{y:.1f} kW<extra></extra>'
    ),
    row=3, col=1
)

# === Row 4: Prices ===
if day_ahead_prices_corrected is not None:
    fig_improved.add_trace(
        go.Scatter(
            x=timesteps,
            y=day_ahead_prices_corrected,
            name="Day-Ahead Price",
            mode='lines+markers',
            line=dict(color='#d62728', width=2.5),
            marker=dict(size=4, color='#d62728'),
            fill='tozeroy',
            fillcolor='rgba(214, 39, 40, 0.15)',
            hovertemplate='<b>%{x}</b><br>Price: €%{y:.3f}/kWh<extra></extra>'
        ),
        row=4, col=1
    )

# Update y-axes labels and ranges
fig_improved.update_yaxes(title_text="Power (kW)", row=1, col=1)
fig_improved.update_yaxes(title_text="Power (kW)", row=2, col=1)
fig_improved.update_yaxes(title_text="Power (kW)", row=3, col=1)
if day_ahead_prices_corrected is not None:
    fig_improved.update_yaxes(title_text="Price (€/kWh)", row=4, col=1)

# Set appropriate y-axis range for grid power to show small variations
grid_power_min = grid_power.min()
grid_power_max = grid_power.max()
grid_power_margin = (grid_power_max - grid_power_min) * 0.1  # 10% margin
fig_improved.update_yaxes(
    range=[grid_power_min - grid_power_margin, grid_power_max + grid_power_margin],
    row=1, col=1
)

fig_improved.update_xaxes(title_text="Time", row=4, col=1)

fig_improved.update_layout(
    title="Energy Dispatch and Prices (Separate Scales)",
    height=1000,
    hovermode='x unified',
    showlegend=True,
    template='plotly_white'
)

fig_improved.show()

print(f"\n✓ Improved visualization created with separate scales:")
print(f"  Grid Power range: {grid_power_min:.0f} - {grid_power_max:.0f} kW (zoomed to show variations)")
print(f"  PV scale: 0 - {pv_dispatch.max():.0f} kW")
print(f"  Battery scale: {battery_dispatch.min():.0f} - {battery_dispatch.max():.0f} kW")


✓ Improved visualization created with separate scales:
  Grid Power range: 24333 - 25439 kW (zoomed to show variations)
  PV scale: 0 - 605 kW
  Battery scale: -1471 - 1312 kW


In [345]:

# ===== Correlation Visualization Alternatives =====
from scipy import stats

# Ensure both arrays have same length
min_len = min(len(battery_dispatch), len(day_ahead_prices_corrected))
battery_trunc = battery_dispatch[:min_len]
price_trunc = day_ahead_prices_corrected[:min_len]
time_trunc = timesteps[:min_len]

# ===== ALTERNATIVE 2: Normalized Dual-Axis =====
batt_norm = (battery_trunc - battery_trunc.min()) / (battery_trunc.max() - battery_trunc.min())
price_norm = (price_trunc - price_trunc.min()) / (price_trunc.max() - price_trunc.min())

fig_dual = make_subplots(specs=[[{"secondary_y": True}]])
fig_dual.add_trace(
    go.Scatter(x=time_trunc, y=price_norm, name="Price", line=dict(color='#d62728', width=2.5)),
    secondary_y=False
)
fig_dual.add_trace(
    go.Scatter(x=time_trunc, y=batt_norm, name="Battery", line=dict(color='#2ca02c', width=2.5)),
    secondary_y=True
)
fig_dual.update_yaxes(title_text="Normalized Price", secondary_y=False)
fig_dual.update_yaxes(title_text="Normalized Battery", secondary_y=True)
fig_dual.update_layout(
    title="Dual-Axis: Direct temporal comparison (both normalized 0-1)",
    height=500,
    hovermode='x unified',
    template='plotly_white'
)
fig_dual.show()


In [346]:

# ===== Extract Residential Demand Data =====
# Extract residential demand using the transmission link flow_out (power delivered to residential demand)
try:
    # Get all transmission-to-residential-demand techs
    all_techs = model.results.techs.values
    res_demand_techs = [t for t in all_techs if 'residential_demand' in t.lower()]
    
    if res_demand_techs:
        # Extract flow_out from the transmission links to residential demand
        residential_demand_data = model.results.flow_out.sel(techs=res_demand_techs, carriers="electricity").sum(dim='techs')
        # Sum across all nodes to get total residential demand
        residential_demand_data = residential_demand_data.sum(dim='nodes')
        residential_demand = residential_demand_data.to_series().dropna().values
        print(f"Residential Demand (from transmission flows): min={residential_demand.min():.1f}, max={residential_demand.max():.1f} kW")
    else:
        residential_demand = None
        print("No residential demand techs found")
except Exception as e:
    print(f"Error extracting residential demand: {e}")
    import traceback
    traceback.print_exc()
    residential_demand = None

# Battery dispatch (already extracted earlier)
print(f"Battery Dispatch: min={battery_dispatch.min():.1f}, max={battery_dispatch.max():.1f} kW")

# Verify data availability
if residential_demand is not None:
    min_len_rbd = min(len(residential_demand), len(battery_dispatch))
    residential_demand_trunc = residential_demand[:min_len_rbd]
    battery_dispatch_trunc = battery_dispatch[:min_len_rbd]
    time_trunc_rbd = timesteps[:min_len_rbd]
    print(f"\nData aligned for analysis: {min_len_rbd} timesteps")


Residential Demand (from transmission flows): min=541.0, max=1083.8 kW
Battery Dispatch: min=-1471.3, max=1312.3 kW

Data aligned for analysis: 336 timesteps


In [347]:

# ===== Normalized Dual-Axis Plot: Direct Temporal Comparison =====

if residential_demand is not None:
    # Normalize both signals to 0-1 range
    demand_norm = (residential_demand_trunc - residential_demand_trunc.min()) / (residential_demand_trunc.max() - residential_demand_trunc.min())
    batt_norm_rbd = (battery_dispatch_trunc - battery_dispatch_trunc.min()) / (battery_dispatch_trunc.max() - battery_dispatch_trunc.min())

    fig_dual_rbd = make_subplots(specs=[[{"secondary_y": True}]])
    
    fig_dual_rbd.add_trace(
        go.Scatter(
            x=time_trunc_rbd, 
            y=demand_norm, 
            name="Residential Demand", 
            line=dict(color='#9467bd', width=2.5),
            fill='tozeroy',
            fillcolor='rgba(148, 103, 189, 0.2)'
        ),
        secondary_y=False
    )
    
    fig_dual_rbd.add_trace(
        go.Scatter(
            x=time_trunc_rbd, 
            y=batt_norm_rbd, 
            name="Battery Dispatch", 
            line=dict(color='#2ca02c', width=2.5),
            fill='tozeroy',
            fillcolor='rgba(44, 160, 44, 0.2)'
        ),
        secondary_y=True
    )
    
    fig_dual_rbd.update_yaxes(title_text="Normalized Demand (0-1)", secondary_y=False)
    fig_dual_rbd.update_yaxes(title_text="Normalized Battery (0-1)", secondary_y=True)
    
    fig_dual_rbd.update_layout(
        title="Dual-Axis: Normalized Residential Demand vs Battery Dispatch (direct temporal comparison)",
        height=500,
        hovermode='x unified',
        template='plotly_white'
    )
    
    fig_dual_rbd.show()
    print("✓ Normalized dual-axis plot created")


✓ Normalized dual-axis plot created
